In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def pack_upper_triangular(U: torch.Tensor, n: int) -> torch.Tensor:
    """
    U: (..., m) where m = n(n+1)/2
    returns Omega: (..., n, n) upper triangular filled row-wise
    """
    *batch, m = U.shape
    assert m == n * (n + 1) // 2
    Omega = U.new_zeros(*batch, n, n)
    idx = 0
    for i in range(n):
        for j in range(i, n):
            Omega[..., i, j] = U[..., idx]
            idx += 1
    return Omega

def gaussian_loglik(r: torch.Tensor, Sigma: torch.Tensor, jitter: float = 1e-6) -> torch.Tensor:
    """
    r: (..., n)
    Sigma: (..., n, n) SPD
    returns log p(r | 0, Sigma) with batch dims
    """
    n = r.shape[-1]
    I = torch.eye(n, device=Sigma.device, dtype=Sigma.dtype)
    Sigma_j = Sigma + jitter * I

    L = torch.linalg.cholesky(Sigma_j)  # (..., n, n)
    # Solve Sigma^{-1} r via Cholesky
    # y = L^{-1} r, then quad = y^T y
    y = torch.linalg.solve_triangular(L, r.unsqueeze(-1), upper=False)  # (..., n, 1)
    quad = (y.squeeze(-1) ** 2).sum(dim=-1)  # (...)

    logdet = 2.0 * torch.log(torch.diagonal(L, dim1=-2, dim2=-1)).sum(dim=-1)  # (...)
    return -0.5 * (n * math.log(2 * math.pi) + logdet + quad)

class NeuralDiagBEKK(nn.Module):
    """
    Neural diagonal-BEKK(1,1):
      Sigma_t = Omega_t^T Omega_t + A_t^T r_{t-1} r_{t-1}^T A_t + B_t^T Sigma_{t-1} B_t
    where Omega_t upper-triangular, A_t,B_t diagonal (a_t,b_t in (0,1)).
    """
    def __init__(self, n_assets: int, hidden_size: int = 32, mlp_size: int = 64):
        super().__init__()
        self.n = n_assets
        self.hidden_size = hidden_size

        self.gru = nn.GRU(input_size=self.n, hidden_size=hidden_size, batch_first=True)

        # output dim: m + n + n, where m = n(n+1)/2
        self.m = self.n * (self.n + 1) // 2
        out_dim = self.m + 2 * self.n

        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, mlp_size),
            nn.Tanh(),
            nn.Linear(mlp_size, out_dim),
        )

        # optional: learn a global scale for Omega part (stabilisiert oft)
        self.omega_scale = nn.Parameter(torch.tensor(0.1))

    def forward(self, returns: torch.Tensor, Sigma0: torch.Tensor = None):
        """
        returns: (B, T, n) or (T, n) (will be expanded to batch=1)
        Sigma0: (B, n, n) or (n, n) optional initial covariance
        Output:
          Sigmas: (B, T, n, n) covariance for each time t (aligned with r_t)
        Convention here:
          We use r_{t-1} to build Sigma_t. So for t=0, we use Sigma0 and r_{-1} not available;
          Practically, we set Sigma_0 = Sigma0 and start recursion at t=1.
        """
        if returns.dim() == 2:
            returns = returns.unsqueeze(0)  # (1, T, n)
        B, T, n = returns.shape
        assert n == self.n

        device = returns.device
        dtype = returns.dtype

        if Sigma0 is None:
            # simple default: identity
            Sigma0 = torch.eye(n, device=device, dtype=dtype).unsqueeze(0).repeat(B, 1, 1)
        else:
            if Sigma0.dim() == 2:
                Sigma0 = Sigma0.unsqueeze(0).repeat(B, 1, 1)
            assert Sigma0.shape == (B, n, n)

        # GRU over returns up to t-1; easiest: feed full sequence and use h_t aligned
        h_seq, _ = self.gru(returns)  # (B, T, hidden)

        Sigmas = returns.new_zeros(B, T, n, n)
        Sigmas[:, 0] = Sigma0

        # recursion from t=1..T-1 using (h_{t-1}, r_{t-1}, Sigma_{t-1})
        for t in range(1, T):
            ht_minus_1 = h_seq[:, t-1, :]  # (B, hidden)
            theta = self.mlp(ht_minus_1)   # (B, m+2n)

            omega_flat = theta[:, :self.m]               # (B, m)
            a_raw = theta[:, self.m:self.m + n]          # (B, n)
            b_raw = theta[:, self.m + n:self.m + 2*n]    # (B, n)

            # constraints
            # a,b in (0,1) to limit explosiveness
            a = torch.sigmoid(a_raw)
            b = torch.sigmoid(b_raw)

            Omega = pack_upper_triangular(omega_flat, n)  # (B, n, n)
            Omega = self.omega_scale * Omega

            r_prev = returns[:, t-1, :]  # (B, n)
            Sigma_prev = Sigmas[:, t-1, :, :]  # (B, n, n)

            # compute A^T r r^T A with diagonal A
            # Equivalent: (a ⊙ r)(a ⊙ r)^T
            ar = a * r_prev  # (B, n)
            shock = ar.unsqueeze(-1) @ ar.unsqueeze(-2)  # (B, n, n)

            # compute B^T Sigma B with diagonal B: elementwise scaling
            # (B diag) * Sigma * (B diag) => Sigma_ij * b_i * b_j
            persist = Sigma_prev * (b.unsqueeze(-1) * b.unsqueeze(-2))  # (B, n, n)

            base = Omega.transpose(-1, -2) @ Omega  # (B, n, n)

            Sigmas[:, t] = base + shock + persist

        return Sigmas

    def neg_loglik(self, returns: torch.Tensor, Sigmas: torch.Tensor) -> torch.Tensor:
        """
        returns: (B, T, n)
        Sigmas:  (B, T, n, n)
        we evaluate loglik for t=1..T-1 using Sigma_t for r_t (common convention)
        """
        if returns.dim() == 2:
            returns = returns.unsqueeze(0)
        B, T, n = returns.shape
        # skip t=0 because Sigma_0 is initialization
        ll = gaussian_loglik(returns[:, 1:, :], Sigmas[:, 1:, :, :])  # (B, T-1)
        return -ll.mean()  # average negative loglik